# Total Part Risk Score (TPRS) — Root Cause Analysis Engine
### (LLM-enhanced variant — deterministic core + optional guarded LLM polish)

## 1. Project Overview

This notebook explains, in business language, **why a part received the
Total Part Risk Score (TPRS) it did**. It does not recompute TPRS, the
AHP weights, the normalization, or the hard-stop rule — those are
reused exactly as produced by the upstream `Total_Part_Risk_Score_v4`
notebook. This notebook's only job is explainability.

**Formula extracted from the source notebook (v4), reused as-is:**

$$
L = \tfrac{2}{3} \cdot N_{Compliance} + \tfrac{1}{3} \cdot N_{Manufacturer}
$$
$$
I = 0.75 \cdot N_{Alternative} + 0.25 \cdot N_{Stock}
$$
$$
TPRS = \frac{L \times I}{100}, \quad \text{forced to 100 if Compliance risk = High Risk AND Alternative risk = Critical (the "hard-stop" rule)}
$$

Both AHP weight sets are reproduced here using the exact same pairwise
comparison matrices as the source notebook (Section 8), not the rounded
0.667/0.333 display values — so every number in this notebook's root
cause explanations reconciles exactly back to `L` and `I`.

**Inventory Risk and Distributor Risk are intentionally excluded from
TPRS** (per the source notebook: Inventory is reported separately by
design; Distributor Risk is excluded pending data provenance). This
notebook stays scoped to exactly the four factors TPRS is actually
built from — Compliance, Manufacturer, Alternative, and Stock — and
does not reference Inventory Risk anywhere.

## 2. Notebook Objectives

- Load the TPRS v4 output exactly as produced upstream, plus enrich it
  with the specific supporting detail already computed by the Stock and
  Alternative risk notebooks (main risk drivers, verified alternative
  counts) so explanations reference real, specific numbers.
- Decompose `L` and `I` into the exact contribution of each of the four
  factors (Compliance, Manufacturer, Alternative, Stock) — an exact
  mathematical reconstruction, not an approximation.
- Grade each factor's contribution using the **same absolute band
  thresholds TPRS itself uses** (`classify()`: <20/40/60/80), so no new
  or arbitrary thresholds are introduced anywhere in this notebook.
- Treat the hard-stop rule as a first-class, always-stated business
  rule override, not just another ranked contributor — because in the
  current dataset, every single Critical-category part is hard-stop
  driven (confirmed in Section 18).
- Validate every record, batch process every High/Critical part, and
  export a Root Cause Report per part.

## 3. Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
cd /content/drive/MyDrive/SiaCore

/content/drive/MyDrive/SiaCore


In [ ]:
from __future__ import annotations

import logging
import warnings
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
from pathlib import Path
from typing import Callable, Dict, List, Optional

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

## 4. Configuration

The primary input is the TPRS v4 export. Two more files are joined in
purely to enrich explanations with real supporting detail — the Stock
and Alternative notebooks already compute specific, per-part detail
(main risk driver, verified alternative count) that's more useful in an
explanation than the normalized score alone. Nothing outside the four
factors TPRS is built from (Compliance, Manufacturer, Alternative,
Stock) is loaded or referenced.

In [ ]:
# =============================================================================
# Project Configuration
# =============================================================================

PROJECT_NAME = "TPRS Root Cause Analysis Engine"
VERSION = "1.0.0"

GENERATED_AT = datetime.now()
SCORE_DETECTED_DATE = GENERATED_AT.strftime("%Y-%m-%d")

# Primary input: the TPRS v4 output, reused exactly as produced upstream.
INPUT_FILE_CANDIDATES = [
    Path("total_part_risk_score_v4.csv"),
    Path("/mnt/user-data/uploads/total_part_risk_score_v4.csv"),
]

# Enrichment inputs: joined on `mpn` purely to ground explanations in real
# supporting detail. Never used to recompute TPRS itself.
STOCK_ENRICHMENT_CANDIDATES = [
    Path("stock_risk_rows__2_.csv"),
    Path("/mnt/user-data/uploads/stock_risk_rows__2_.csv"),
]
ALT_ENRICHMENT_CANDIDATES = [
    Path("alternatives_risk_rows.csv"),
    Path("/mnt/user-data/uploads/alternatives_risk_rows.csv"),
]

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
EXPORT_FILE = OUTPUT_DIR / "total_part_risk_root_cause_analysis.csv"

# part_category values in scope for Root Cause Analysis. In the current
# dataset only Critical and Very Low actually occur at scale (see Section
# 18), but this stays configurable rather than hardcoded to what's
# currently in the data.
HIGH_RISK_CATEGORIES = ["High", "Critical"]
KNOWN_CATEGORIES = ["Very Low", "Low", "Medium", "High", "Critical"]

REQUIRED_COLUMNS = ["mpn", "N_STOCK", "N_MFR", "N_ALT", "N_COMP", "L", "I",
                     "part_score", "part_category", "hard_stop"]

MAX_ROOT_CAUSES_PER_RECORD = 4

## 5. Logging

In [ ]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("TPRS_RCA")
logger.info(f"{PROJECT_NAME} v{VERSION} initialized")

## 6. Data Loading, Validation & Enrichment

The TPRS output is loaded and checked for the columns this engine
depends on. It's then left-joined with the Stock and Alternative
enrichment files (both keyed on `mpn`, both verified duplicate-free).
No other dataset is loaded — this notebook stays scoped to the four
factors TPRS is actually built from.

In [ ]:
def resolve_input_file(candidates: List[Path]) -> Path:
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f"None of the candidate files were found: {candidates}")


INPUT_FILE = resolve_input_file(INPUT_FILE_CANDIDATES)
tprs_df = pd.read_csv(INPUT_FILE)

missing_columns = [c for c in REQUIRED_COLUMNS if c not in tprs_df.columns]
if missing_columns:
    raise ValueError(f"Input file is missing required columns: {missing_columns}")

logger.info(f"Loaded '{INPUT_FILE}' — {len(tprs_df):,} parts")

In [ ]:
# =============================================================================
# Enrichment — joined for real supporting detail, never for scoring
# =============================================================================

stock_path = resolve_input_file(STOCK_ENRICHMENT_CANDIDATES)
stock_enrich = pd.read_csv(stock_path)[
    ["mpn", "Final_Market_Risk_Score", "risk_category", "Main_Risk_Driver", "Secondary_Risk_Driver"]
].rename(columns={
    "Final_Market_Risk_Score": "stock_final_score",
    "risk_category": "stock_risk_category",
    "Main_Risk_Driver": "stock_main_driver",
    "Secondary_Risk_Driver": "stock_secondary_driver",
})

alt_path = resolve_input_file(ALT_ENRICHMENT_CANDIDATES)
alt_enrich = pd.read_csv(alt_path)[
    ["mpn", "n_real_alts", "best_ASR", "redundancy_credit", "PARS", "PARS_category"]
].rename(columns={
    "n_real_alts": "alt_n_real_alts", "best_ASR": "alt_best_asr",
    "redundancy_credit": "alt_redundancy_credit", "PARS": "alt_pars", "PARS_category": "alt_pars_category",
})

part_df = (
    tprs_df
    .merge(stock_enrich, on="mpn", how="left")
    .merge(alt_enrich, on="mpn", how="left")
)

logger.info(f"Enriched with Stock/Alternative detail for {len(part_df):,} parts")
part_df[["mpn", "N_STOCK", "N_MFR", "N_ALT", "N_COMP", "part_score", "part_category", "hard_stop"]].head(5)

,mpn,N_STOCK,N_MFR,N_ALT,N_COMP,part_score,part_category,hard_stop
0,"TFA9879HN/N1,118",46.721170,10.000000,96,87.5,100.0,Critical,True
1,TCM320AC37CN,42.652324,4.783542,96,85.0,100.0,Critical,True
2,TCM320AC36CPT,89.089196,4.783542,96,85.0,100.0,Critical,True
3,AD95231BCPZ,85.866834,4.783542,96,85.0,100.0,Critical,True
4,AB55703HCHCFLCT,85.866834,1.925504,96,85.0,100.0,Critical,True


## 7. Extracted Formula Reference *(reproduced from the source notebook)*

The AHP weights below are re-derived using the **exact same pairwise
comparison matrices** as the TPRS v4 notebook — not copied as rounded
constants — so that every axis-share number in this notebook's
explanations reconstructs `L` and `I` exactly. This is verified in
Section 18.

In [ ]:
# =============================================================================
# AHP Weight Reproduction (identical method to the source notebook)
# =============================================================================

def ahp_solve(M: np.ndarray, names: List[str]) -> Dict[str, float]:
    col_norm = M / M.sum(axis=0)
    w = col_norm.mean(axis=1)
    return dict(zip(names, w))


_w_L = ahp_solve(np.array([[1, 2], [1/2, 1]]), ["Compliance", "Manufacturer"])
_w_I = ahp_solve(np.array([[1, 3], [1/3, 1]]), ["Alternative", "Stock"])

L_WEIGHTS = {"Compliance": _w_L["Compliance"], "Manufacturer": _w_L["Manufacturer"]}
I_WEIGHTS = {"Alternative": _w_I["Alternative"], "Stock": _w_I["Stock"]}

assert abs(sum(L_WEIGHTS.values()) - 1.0) < 1e-9
assert abs(sum(I_WEIGHTS.values()) - 1.0) < 1e-9

print("Likelihood (L) weights:", {k: round(v, 6) for k, v in L_WEIGHTS.items()})
print("Impact (I) weights:    ", {k: round(v, 6) for k, v in I_WEIGHTS.items()})

Likelihood (L) weights: {'Compliance': np.float64(0.666667), 'Manufacturer': np.float64(0.333333)}
Impact (I) weights:     {'Alternative': np.float64(0.75), 'Stock': np.float64(0.25)}


In [ ]:
# =============================================================================
# Absolute band thresholds — identical to the source notebook's classify()
# =============================================================================

def classify(score: float) -> str:
    """Reproduced verbatim from the TPRS v4 notebook. Used here both to
    sanity-check part_category and to grade individual factor severity on
    the same absolute 0-100 scale, so no new thresholds are invented."""
    if score < 20: return "Very Low"
    if score < 40: return "Low"
    if score < 60: return "Medium"
    if score < 80: return "High"
    return "Critical"


# Sanity check: reclassifying part_score with the reproduced function must
# match the part_category already in the file, for every non-hard-stop row
# (hard-stop rows are forced to 100/Critical by a separate rule).
_check = part_df[~part_df["hard_stop"]].apply(lambda r: classify(r["part_score"]) == r["part_category"], axis=1)
assert _check.all(), "Reproduced classify() disagrees with the source file's part_category"
logger.info("classify() reproduction verified against every non-hard-stop record.")

## 8. Factor Knowledge Base

Four factors feed TPRS. Each is already a 0-100 risk score (higher is
always riskier), so — unlike a portfolio-relative approach — factor
severity can be graded directly against the same absolute band edges
`classify()` uses for TPRS itself.

In [ ]:
class ContributionLevel(str, Enum):
    CRITICAL = "Critical Contributor"
    HIGH = "High Contributor"
    MODERATE = "Moderate Contributor"
    MINOR = "Minor Contributor"
    NONE = "No Contribution"


_SEVERITY_RANK = {
    ContributionLevel.CRITICAL: 4, ContributionLevel.HIGH: 3,
    ContributionLevel.MODERATE: 2, ContributionLevel.MINOR: 1, ContributionLevel.NONE: 0,
}


def contribution_from_band(value: float) -> ContributionLevel:
    """Grade a normalized 0-100 factor score using the exact same band
    edges as classify() above -- Critical >=80, High >=60, Moderate >=40,
    Minor >=20, else No Contribution."""
    if pd.isna(value):
        return ContributionLevel.NONE
    if value >= 80: return ContributionLevel.CRITICAL
    if value >= 60: return ContributionLevel.HIGH
    if value >= 40: return ContributionLevel.MODERATE
    if value >= 20: return ContributionLevel.MINOR
    return ContributionLevel.NONE

In [ ]:
@dataclass
class FactorKnowledge:
    key: str
    title: str
    axis: str            # "Likelihood" or "Impact"
    axis_weight: float
    business_meaning: str


TPRS_KNOWLEDGE: Dict[str, FactorKnowledge] = {
    "N_COMP": FactorKnowledge(
        key="N_COMP", title="Regulatory Compliance Risk", axis="Likelihood",
        axis_weight=L_WEIGHTS["Compliance"],
        business_meaning=(
            "Exposure to regulatory non-compliance (RoHS/REACH and related "
            "trial results), which drives how likely a compliance-related "
            "disruption is for this part."
        ),
    ),
    "N_MFR": FactorKnowledge(
        key="N_MFR", title="Manufacturer Risk", axis="Likelihood",
        axis_weight=L_WEIGHTS["Manufacturer"],
        business_meaning=(
            "The financial and operational stability of the part's "
            "manufacturer, which drives how likely a supply disruption "
            "originating at the manufacturer is."
        ),
    ),
    "N_ALT": FactorKnowledge(
        key="N_ALT", title="Part Alternative Scarcity", axis="Impact",
        axis_weight=I_WEIGHTS["Alternative"],
        business_meaning=(
            "How many verified substitute parts exist and how good they "
            "are, which determines how severe the impact would be if this "
            "part became unavailable."
        ),
    ),
    "N_STOCK": FactorKnowledge(
        key="N_STOCK", title="Market Stock Risk", axis="Impact",
        axis_weight=I_WEIGHTS["Stock"],
        business_meaning=(
            "Open-market stock availability, lifecycle stage, and supplier "
            "concentration, which determine how severe the impact would be "
            "if this part needed to be resourced quickly."
        ),
    ),
}

pd.DataFrame([
    {"Factor": k.title, "Axis": k.axis, "AHP Weight": round(k.axis_weight, 4), "Business Meaning": k.business_meaning}
    for k in TPRS_KNOWLEDGE.values()
])

,Factor,Axis,AHP Weight,Business Meaning
0,Regulatory Compliance Risk,Likelihood,0.6667,"Exposure to regulatory non-compliance (RoHS/REACH and related trial results), which drives how likely a compliance-r..."
1,Manufacturer Risk,Likelihood,0.3333,"The financial and operational stability of the part's manufacturer, which drives how likely a supply disruption orig..."
2,Part Alternative Scarcity,Impact,0.7500,"How many verified substitute parts exist and how good they are, which determines how severe the impact would be if t..."
3,Market Stock Risk,Impact,0.2500,"Open-market stock availability, lifecycle stage, and supplier concentration, which determine how severe the impact w..."


## 9. Business Language Library

Each factor gets a narrative builder producing a plain-language Human
Explanation, a Business Impact statement, and Recommendations —
enriched with real supporting numbers from the Stock and Alternative
data wherever a part-level match exists, and falling back to the
normalized score alone when it doesn't.

In [ ]:
@dataclass
class FactorNarrative:
    human: str
    business_impact: str
    recommendations: List[str]

In [ ]:
def build_compliance_narrative(record: pd.Series, contribution: ContributionLevel) -> FactorNarrative:
    score = record["N_COMP"]
    human = (
        f"This part carries a regulatory compliance risk score of {score:.0f} "
        f"out of 100, based on its RoHS/REACH standing and related trial results."
    )
    impact = {
        ContributionLevel.CRITICAL: "Compliance exposure is severe enough that a regulatory issue (an import block, a customer rejection, or a required redesign) is a real near-term possibility.",
        ContributionLevel.HIGH: "Compliance exposure is high enough to warrant proactive review before this part is designed into new products.",
        ContributionLevel.MODERATE: "There is a moderate amount of compliance exposure worth keeping on file.",
        ContributionLevel.MINOR: "Compliance exposure is only slightly elevated.",
        ContributionLevel.NONE: "Compliance is not a concern for this part right now.",
    }[contribution]
    recs = {
        ContributionLevel.CRITICAL: ["Escalate to compliance/quality for a formal review", "Confirm current RoHS/REACH certificates before further use", "Evaluate whether this part can be replaced with a compliant equivalent"],
        ContributionLevel.HIGH: ["Request updated compliance documentation from the manufacturer", "Flag this part for compliance review before new designs"],
        ContributionLevel.MODERATE: ["Keep compliance documentation current for this part"],
        ContributionLevel.MINOR: ["No immediate action required"],
        ContributionLevel.NONE: ["No action required"],
    }[contribution]
    return FactorNarrative(human, impact, recs)

In [ ]:
def build_manufacturer_narrative(record: pd.Series, contribution: ContributionLevel) -> FactorNarrative:
    score = record["N_MFR"]
    human = (
        f"The part's manufacturer carries a stability risk score of {score:.0f} "
        f"out of 100, reflecting its financial and operational track record."
    )
    impact = {
        ContributionLevel.CRITICAL: "There's a real chance of a supply disruption originating at the manufacturer itself, independent of this specific part's own availability.",
        ContributionLevel.HIGH: "The manufacturer's stability is a genuine concern and worth factoring into sourcing decisions.",
        ContributionLevel.MODERATE: "The manufacturer carries a moderate amount of stability risk.",
        ContributionLevel.MINOR: "Manufacturer risk is only slightly elevated.",
        ContributionLevel.NONE: "The manufacturer is not a concern for this part right now.",
    }[contribution]
    recs = {
        ContributionLevel.CRITICAL: ["Review the manufacturer's financial/operational standing", "Identify a second-source manufacturer for this part"],
        ContributionLevel.HIGH: ["Monitor the manufacturer's stability going forward", "Explore a second source if one doesn't already exist"],
        ContributionLevel.MODERATE: ["Keep an eye on manufacturer news as part of routine review"],
        ContributionLevel.MINOR: ["No immediate action required"],
        ContributionLevel.NONE: ["No action required"],
    }[contribution]
    return FactorNarrative(human, impact, recs)

In [ ]:
def build_alternative_narrative(record: pd.Series, contribution: ContributionLevel) -> FactorNarrative:
    score = record["N_ALT"]
    n_alts = record.get("alt_n_real_alts")
    best_asr = record.get("alt_best_asr")
    if pd.notna(n_alts) and pd.notna(best_asr):
        n_alts = int(n_alts)
        if n_alts == 0:
            human = (
                "There are currently no verified alternative parts for this "
                "component, so if it becomes unavailable there is no fallback in place."
            )
        else:
            human = (
                f"There are {n_alts} verified alternative part(s) for this "
                f"component, and the best one scores {best_asr:.0f} out of 100 as a substitute."
            )
    else:
        human = (
            f"This part scores {score:.0f} out of 100 on alternative scarcity, "
            f"reflecting how few good substitutes are available."
        )
    impact = {
        ContributionLevel.CRITICAL: "If this part becomes unavailable, there is essentially no fallback, so the impact on production would be severe and immediate.",
        ContributionLevel.HIGH: "Substitute options are thin, so losing this part would be difficult to work around quickly.",
        ContributionLevel.MODERATE: "There are some substitute options, but not enough to fully offset losing this part.",
        ContributionLevel.MINOR: "Substitute options exist and would likely soften the impact of losing this part.",
        ContributionLevel.NONE: "Good substitute options exist, so losing this part would be manageable.",
    }[contribution]
    recs = {
        ContributionLevel.CRITICAL: ["Prioritize qualifying an alternative part immediately", "Evaluate a redesign to a more available component"],
        ContributionLevel.HIGH: ["Qualify at least one alternative part for this component"],
        ContributionLevel.MODERATE: ["Keep the existing alternative options qualified and current"],
        ContributionLevel.MINOR: ["No immediate action required"],
        ContributionLevel.NONE: ["No action required"],
    }[contribution]
    return FactorNarrative(human, impact, recs)

In [ ]:
def build_stock_narrative(record: pd.Series, contribution: ContributionLevel) -> FactorNarrative:
    score = record["N_STOCK"]
    main_driver = record.get("stock_main_driver")
    secondary_driver = record.get("stock_secondary_driver")
    if pd.notna(main_driver):
        driver_text = f"driven mainly by {main_driver}"
        if pd.notna(secondary_driver):
            driver_text += f", with {secondary_driver} as a secondary factor"
        human = f"Open-market stock conditions for this part are {driver_text}."
    else:
        human = (
            f"This part scores {score:.0f} out of 100 on market stock risk, "
            f"reflecting open-market availability and lifecycle stage."
        )
    impact = {
        ContributionLevel.CRITICAL: "If this part needed to be resourced quickly on the open market, availability conditions right now make that very difficult.",
        ContributionLevel.HIGH: "Resourcing this part quickly on the open market would be challenging under current conditions.",
        ContributionLevel.MODERATE: "Open-market conditions add some friction to resourcing this part quickly.",
        ContributionLevel.MINOR: "Open-market conditions are only slightly less favorable than usual.",
        ContributionLevel.NONE: "Open-market conditions for this part are healthy right now.",
    }[contribution]
    recs = {
        ContributionLevel.CRITICAL: ["Monitor lifecycle status and open-market availability closely", "Consider a last-time-buy if lifecycle status is end-of-life"],
        ContributionLevel.HIGH: ["Track this part's lifecycle stage and market availability"],
        ContributionLevel.MODERATE: ["Check market conditions periodically as part of routine review"],
        ContributionLevel.MINOR: ["No immediate action required"],
        ContributionLevel.NONE: ["No action required"],
    }[contribution]
    return FactorNarrative(human, impact, recs)

In [ ]:
NARRATIVE_BUILDERS: Dict[str, Callable[[pd.Series, ContributionLevel], FactorNarrative]] = {
    "N_COMP": build_compliance_narrative,
    "N_MFR": build_manufacturer_narrative,
    "N_ALT": build_alternative_narrative,
    "N_STOCK": build_stock_narrative,
}
assert set(NARRATIVE_BUILDERS) == set(TPRS_KNOWLEDGE)
logger.info("Business Language Library ready.")

## 10. Contribution Engine — Exact Axis-Share Decomposition

Each factor's contribution to `L` or `I` is computed as
`axis_weight × normalized_value`. Because `L` and `I` are themselves
just the weighted sum of two such terms, these axis shares are an
**exact decomposition** — they sum precisely back to `L` and `I`, not
an approximation of influence. Severity is then graded per factor using
the Section 8 band thresholds.

In [ ]:
@dataclass
class FactorContribution:
    key: str
    title: str
    axis: str
    axis_weight: float
    raw_value: float
    axis_share: float
    contribution: ContributionLevel
    severity_score: float


def analyze_factor_contribution(record: pd.Series, key: str) -> FactorContribution:
    knowledge = TPRS_KNOWLEDGE[key]
    value = record[key]
    axis_share = knowledge.axis_weight * value
    contribution = contribution_from_band(value)
    severity_score = knowledge.axis_weight * _SEVERITY_RANK[contribution]
    return FactorContribution(
        key=key, title=knowledge.title, axis=knowledge.axis, axis_weight=knowledge.axis_weight,
        raw_value=value, axis_share=axis_share, contribution=contribution, severity_score=severity_score,
    )


def analyze_record_contributions(record: pd.Series) -> List[FactorContribution]:
    return [analyze_factor_contribution(record, key) for key in TPRS_KNOWLEDGE]

In [ ]:
# Demonstration on a hard-stop Critical part
demo_record = part_df[part_df["hard_stop"]].iloc[0]
demo_contributions = analyze_record_contributions(demo_record)

pd.DataFrame([
    {"Factor": c.title, "Axis": c.axis, "Value (0-100)": round(c.raw_value, 1),
     "Axis Share": round(c.axis_share, 2), "Contribution": c.contribution.value}
    for c in sorted(demo_contributions, key=lambda c: -c.severity_score)
])

,Factor,Axis,Value (0-100),Axis Share,Contribution
0,Part Alternative Scarcity,Impact,96.0,72.00,Critical Contributor
1,Regulatory Compliance Risk,Likelihood,87.5,58.33,Critical Contributor
2,Market Stock Risk,Impact,46.7,11.68,Moderate Contributor
3,Manufacturer Risk,Likelihood,10.0,3.33,No Contribution


## 11. Root Cause Selection & Business Explanation Generator

Root causes are every Critical/High contributor, falling back to the
strongest Moderate contributors if none exist. **The hard-stop rule is
handled separately and always leads the explanation when it applies** —
it is a documented business rule override, not a ranked contributor, and
burying it among four ranked factors would understate why the part is
actually at maximum risk.

In [ ]:
MAX_ROOT_CAUSES = 4  # re-affirmed here from Configuration for local clarity

def select_root_causes(contributions: List[FactorContribution]) -> List[FactorContribution]:
    ranked = sorted(contributions, key=lambda c: -c.severity_score)
    causes = [c for c in ranked if c.contribution in (ContributionLevel.CRITICAL, ContributionLevel.HIGH)]
    if not causes:
        causes = [c for c in ranked if c.contribution == ContributionLevel.MODERATE][:2]
    return causes[:MAX_ROOT_CAUSES]

In [ ]:
def build_explanation_bundle(record: pd.Series, root_causes: List[FactorContribution]):
    hard_stop = bool(record.get("hard_stop", False))

    narratives = [(c, NARRATIVE_BUILDERS[c.key](record, c.contribution)) for c in root_causes]
    if narratives:
        human_explanation = " ".join(n.human for _, n in narratives)
        business_impact = " ".join(n.business_impact for _, n in narratives)
        seen, recommendations = set(), []
        for _, n in narratives:
            for r in n.recommendations:
                if r != "No action required" and r not in seen:
                    seen.add(r); recommendations.append(r)
        if not recommendations:
            recommendations = ["Continue routine monitoring"]
    else:
        human_explanation = "No individual risk factor stands out as a significant driver for this part based on the current data."
        business_impact = "No material impact identified from any single factor."
        recommendations = ["Continue routine monitoring"]

    cause_titles = [c.title for c in root_causes]

    if hard_stop:
        hard_stop_sentence = (
            "This part is capped at the maximum risk score because it triggered the "
            "documented hard-stop rule: Compliance risk is High Risk AND Alternative "
            "risk is Critical at the same time. When both are true, the part is "
            "treated as maximum risk regardless of what the individual factor scores "
            "would otherwise suggest."
        )
        human_explanation = hard_stop_sentence + " " + human_explanation
        recommendations = [
            "Resolve the compliance flag or qualify a verified alternative part -- "
            "either one on its own would lift this part out of the hard-stop rule",
        ] + recommendations
        cause_titles = ["Hard-Stop Rule (Compliance + Alternative)"] + cause_titles

    return human_explanation, business_impact, recommendations, cause_titles

## 11B. Priority Classification

A simple, deterministic action-urgency label. A hard-stop is always
`Immediate` — it represents the two most severe possible conditions
occurring together, independent of everything else.

In [ ]:
def determine_priority(part_category: str, hard_stop: bool, root_causes: List[FactorContribution]) -> str:
    if hard_stop:
        return "Immediate"
    if part_category not in ("High", "Critical"):
        return "Monitor"
    severe = sum(1 for c in root_causes if c.contribution in (ContributionLevel.CRITICAL, ContributionLevel.HIGH))
    if part_category == "Critical":
        return "Immediate" if severe >= 1 else "High"
    return "High" if severe >= 1 else "Medium"

## 12. Recommendations Grouped by Urgency

Each recommendation is tagged by the real severity of the root cause
behind it: `Immediate` for the hard-stop rule or a Critical Contributor,
`Short-term` for a High Contributor, `Monitor` otherwise.

In [ ]:
_URGENCY_LABELS = {
    ContributionLevel.CRITICAL: "Immediate",
    ContributionLevel.HIGH: "Short-term",
    ContributionLevel.MODERATE: "Monitor",
    ContributionLevel.MINOR: "Monitor",
    ContributionLevel.NONE: "Monitor",
}


def group_recommendations_by_urgency(record: pd.Series, root_causes: List[FactorContribution]) -> List[str]:
    hard_stop = bool(record.get("hard_stop", False))
    labeled: List[str] = []
    if hard_stop:
        labeled.append(
            "[Immediate] Resolve the compliance flag or qualify a verified alternative "
            "part -- either one on its own would lift this part out of the hard-stop rule"
        )
    seen = set(labeled)
    for cause in sorted(root_causes, key=lambda c: -c.severity_score):
        narrative = NARRATIVE_BUILDERS[cause.key](record, cause.contribution)
        tag = _URGENCY_LABELS[cause.contribution]
        for rec in narrative.recommendations:
            entry = f"[{tag}] {rec}"
            if rec != "No action required" and entry not in seen:
                seen.add(entry); labeled.append(entry)
    return labeled or ["[Monitor] Continue routine monitoring"]

## 13. Executive Summary Generator

Leads with the hard-stop condition when it applies; otherwise names the
real root-cause factors.

In [ ]:
def generate_executive_summary(record: pd.Series, root_causes: List[FactorContribution], cause_titles: List[str]) -> str:
    hard_stop = bool(record.get("hard_stop", False))
    mpn = record.get("mpn")

    if hard_stop:
        summary = (
            f"Part {mpn} is forced to Critical risk (score 100) by the hard-stop rule, "
            f"because Compliance risk is High Risk and Alternative risk is Critical at "
            f"the same time. "
        )
    elif cause_titles:
        if len(cause_titles) == 1:
            cause_text = cause_titles[0]
        elif len(cause_titles) == 2:
            cause_text = f"{cause_titles[0]} and {cause_titles[1]}"
        else:
            cause_text = ", ".join(cause_titles[:-1]) + f", and {cause_titles[-1]}"
        summary = (
            f"Part {mpn} is currently classified as {record.get('part_category')} risk "
            f"(score {record.get('part_score'):.0f}), primarily driven by {cause_text}. "
        )
    else:
        summary = (
            f"Part {mpn} is currently classified as {record.get('part_category')} risk "
            f"(score {record.get('part_score'):.0f}). "
        )

    summary += "Timely action on these areas is recommended to reduce the risk of a supply disruption."
    return summary

## 14. Part Record Analyzer

Orchestrates every module above for a single part into the final
Root Cause Report.

In [ ]:
@dataclass
class TPRSReport:
    mpn: object
    part_score: float
    part_category: str
    hard_stop: bool
    priority: str
    score_detected_date: str
    detected_root_causes: List[str]
    human_explanation: str
    business_impact: str
    executive_summary: str
    recommendations: List[str]
    recommendations_by_urgency: List[str]
    validation_warnings: List[str] = field(default_factory=list)


def generate_tprs_report(record: pd.Series) -> TPRSReport:
    warns = validate_record(record)
    for w in warns:
        logger.warning(f"[{record.get('mpn', 'unknown')}] {w}")

    contributions = analyze_record_contributions(record)
    root_causes = select_root_causes(contributions)
    human_explanation, business_impact, recommendations, cause_titles = build_explanation_bundle(record, root_causes)
    recommendations_by_urgency = group_recommendations_by_urgency(record, root_causes)
    executive_summary = generate_executive_summary(record, root_causes, cause_titles)

    hard_stop = bool(record.get("hard_stop", False))
    priority = determine_priority(record.get("part_category"), hard_stop, root_causes)

    return TPRSReport(
        mpn=record.get("mpn"), part_score=record.get("part_score"), part_category=record.get("part_category"),
        hard_stop=hard_stop, priority=priority, score_detected_date=SCORE_DETECTED_DATE,
        detected_root_causes=cause_titles, human_explanation=human_explanation,
        business_impact=business_impact, executive_summary=executive_summary,
        recommendations=recommendations, recommendations_by_urgency=recommendations_by_urgency,
        validation_warnings=warns,
    )


def print_tprs_report(report: TPRSReport) -> None:
    print("=" * 78)
    print(f"MPN                : {report.mpn}")
    print(f"Part Score         : {report.part_score:.1f}")
    print(f"Part Category      : {report.part_category}")
    print(f"Hard Stop          : {report.hard_stop}")
    print(f"Priority           : {report.priority}")
    print(f"Score Detected Date: {report.score_detected_date}")
    print("-" * 78)
    print("Detected Root Causes")
    print("-" * 78)
    for c in report.detected_root_causes:
        print(f"  - {c}")
    print("-" * 78)
    print("Human Explanation")
    print("-" * 78)
    print(report.human_explanation)
    print("-" * 78)
    print("Business Impact")
    print("-" * 78)
    print(report.business_impact)
    print("-" * 78)
    print("Executive Summary")
    print("-" * 78)
    print(report.executive_summary)
    print("-" * 78)
    print("Recommended Actions (grouped by urgency)")
    print("-" * 78)
    for r in report.recommendations_by_urgency:
        print(f"  - {r}")
    print("=" * 78)

## 15. Validation Layer

Checks every record before an explanation is generated. Problems are
logged as warnings, never crash the notebook.

In [ ]:
def validate_record(record: pd.Series) -> List[str]:
    warnings_found: List[str] = []
    for col in REQUIRED_COLUMNS:
        if col not in record.index:
            warnings_found.append(f"Missing column: {col}")
            continue
        if pd.isna(record[col]):
            warnings_found.append(f"Null value in {col}")

    for key in TPRS_KNOWLEDGE:
        if key in record.index and pd.notna(record[key]) and not (0 <= record[key] <= 100):
            warnings_found.append(f"{key} outside expected 0-100 range: {record[key]}")

    if "part_category" in record.index and record["part_category"] not in KNOWN_CATEGORIES:
        warnings_found.append(f"Unexpected part_category value: {record['part_category']}")

    if "hard_stop" in record.index and bool(record["hard_stop"]) and record.get("part_score") != 100.0:
        warnings_found.append("hard_stop is True but part_score is not 100 -- inconsistent with the documented rule")

    return warnings_found

In [ ]:
# Demonstration: full report for a hard-stop Critical part
demo_report = generate_tprs_report(demo_record)
print_tprs_report(demo_report)

MPN                : TFA9879HN/N1,118
Part Score         : 100.0
Part Category      : Critical
Hard Stop          : True
Priority           : Immediate
Score Detected Date: 2026-08-01
------------------------------------------------------------------------------
Detected Root Causes
------------------------------------------------------------------------------
  - Hard-Stop Rule (Compliance + Alternative)
  - Part Alternative Scarcity
  - Regulatory Compliance Risk
------------------------------------------------------------------------------
Human Explanation
------------------------------------------------------------------------------
This part is capped at the maximum risk score because it triggered the documented hard-stop rule: Compliance risk is High Risk AND Alternative risk is Critical at the same time. When both are true, the part is treated as maximum risk regardless of what the individual factor scores would otherwise suggest. There are currently no verified alternative par

## 16. Batch Processing

Runs the engine over every part in `HIGH_RISK_CATEGORIES`. Each record
is processed independently and defensively.

In [ ]:
def run_batch_tprs_analysis(data: pd.DataFrame, categories: List[str]) -> pd.DataFrame:
    target = data[data["part_category"].isin(categories)]
    logger.info(f"Running TPRS Root Cause Analysis on {len(target):,} parts (categories: {categories})")

    rows, error_count = [], 0
    for idx, record in target.iterrows():
        try:
            r = generate_tprs_report(record)
            rows.append({
                "MPN": r.mpn,
                "Part Score": r.part_score,
                "Part Category": r.part_category,
                "Hard Stop": r.hard_stop,
                "Priority": r.priority,
                "Score Detected Date": r.score_detected_date,
                "Detected Root Causes": "; ".join(r.detected_root_causes),
                "Human Explanation": r.human_explanation,
                "Business Impact": r.business_impact,
                "Executive Summary": r.executive_summary,
                "Recommendations": "; ".join(r.recommendations),
                "Recommendations (by Urgency)": "; ".join(r.recommendations_by_urgency),
            })
        except Exception as exc:
            error_count += 1
            logger.error(f"Failed to analyze record at index {idx}: {exc}")

    logger.info(f"Batch complete: {len(rows):,} succeeded, {error_count:,} failed")
    return pd.DataFrame(rows)


tprs_results = run_batch_tprs_analysis(part_df, HIGH_RISK_CATEGORIES)
tprs_results.head(10)

,MPN,Part Score,Part Category,Hard Stop,Priority,Score Detected Date,Detected Root Causes,Human Explanation,Business Impact,Executive Summary,Recommendations,Recommendations (by Urgency)
0,"TFA9879HN/N1,118",100.0,Critical,True,Immediate,2026-08-01,Hard-Stop Rule (Compliance + Alternative); Part Alternative Scarcity; Regulatory Compliance Risk,This part is capped at the maximum risk score because it triggered the documented hard-stop rule: Compliance risk is...,"If this part becomes unavailable, there is essentially no fallback, so the impact on production would be severe and ...","Part TFA9879HN/N1,118 is forced to Critical risk (score 100) by the hard-stop rule, because Compliance risk is High ...",Resolve the compliance flag or qualify a verified alternative part -- either one on its own would lift this part out...,[Immediate] Resolve the compliance flag or qualify a verified alternative part -- either one on its own would lift t...
1,TCM320AC37CN,100.0,Critical,True,Immediate,2026-08-01,Hard-Stop Rule (Compliance + Alternative); Part Alternative Scarcity; Regulatory Compliance Risk,This part is capped at the maximum risk score because it triggered the documented hard-stop rule: Compliance risk is...,"If this part becomes unavailable, there is essentially no fallback, so the impact on production would be severe and ...","Part TCM320AC37CN is forced to Critical risk (score 100) by the hard-stop rule, because Compliance risk is High Risk...",Resolve the compliance flag or qualify a verified alternative part -- either one on its own would lift this part out...,[Immediate] Resolve the compliance flag or qualify a verified alternative part -- either one on its own would lift t...
2,TCM320AC36CPT,100.0,Critical,True,Immediate,2026-08-01,Hard-Stop Rule (Compliance + Alternative); Part Alternative Scarcity; Regulatory Compliance Risk; Market Stock Risk,This part is capped at the maximum risk score because it triggered the documented hard-stop rule: Compliance risk is...,"If this part becomes unavailable, there is essentially no fallback, so the impact on production would be severe and ...","Part TCM320AC36CPT is forced to Critical risk (score 100) by the hard-stop rule, because Compliance risk is High Ris...",Resolve the compliance flag or qualify a verified alternative part -- either one on its own would lift this part out...,[Immediate] Resolve the compliance flag or qualify a verified alternative part -- either one on its own would lift t...
3,AD95231BCPZ,100.0,Critical,True,Immediate,2026-08-01,Hard-Stop Rule (Compliance + Alternative); Part Alternative Scarcity; Regulatory Compliance Risk; Market Stock Risk,This part is capped at the maximum risk score because it triggered the documented hard-stop rule: Compliance risk is...,"If this part becomes unavailable, there is essentially no fallback, so the impact on production would be severe and ...","Part AD95231BCPZ is forced to Critical risk (score 100) by the hard-stop rule, because Compliance risk is High Risk ...",Resolve the compliance flag or qualify a verified alternative part -- either one on its own would lift this part out...,[Immediate] Resolve the compliance flag or qualify a verified alternative part -- either one on its own would lift t...
4,AB55703HCHCFLCT,100.0,Critical,True,Immediate,2026-08-01,Hard-Stop Rule (Compliance + Alternative); Part Alternative Scarcity; Regulatory Compliance Risk; Market Stock Risk,This part is capped at the maximum risk score because it triggered the documented hard-stop rule: Compliance risk is...,"If this part becomes unavailable, there is essentially no fallback, so the impact on production would be severe and ...","Part AB55703HCHCFLCT is forced to Critical risk (score 100) by the hard-stop rule, because Compliance risk is High R...",Resolve the compliance flag or qualify a verified alternative part -- either one on its own would lift this part out...,[Immediate] Resolve the compliance flag or qualify a verified alternative part -- 

## 17. CSV Export

In [ ]:
tprs_results.to_csv(EXPORT_FILE, index=False)
logger.info(f"Exported {len(tprs_results):,} TPRS Root Cause reports to '{EXPORT_FILE}'")
tprs_results.shape

(31, 12)

## 18. Engine Reliability & Consistency Checks

No external AI model is called anywhere above this point — every
sentence is a deterministic template. This section verifies the
mathematical and business-rule invariants this notebook depends on.

In [ ]:
# =============================================================================
# Unit Tests — Exact Formula Reconstruction & Business Rules
# =============================================================================

_tests_passed = True

def _check(label: str, condition: bool):
    global _tests_passed
    _tests_passed = _tests_passed and condition
    print(f"{'PASS' if condition else 'FAIL'}  {label}")

# 1. Axis shares must sum exactly to L and I for every record.
_axis_check = part_df.apply(
    lambda r: abs((L_WEIGHTS['Compliance']*r['N_COMP'] + L_WEIGHTS['Manufacturer']*r['N_MFR']) - r['L']) < 1e-6
              and abs((I_WEIGHTS['Alternative']*r['N_ALT'] + I_WEIGHTS['Stock']*r['N_STOCK']) - r['I']) < 1e-6,
    axis=1,
)
_check("Axis-share decomposition reconstructs L and I exactly for every part", bool(_axis_check.all()))

# 2. Band thresholds must match classify() exactly at the boundaries.
_check("contribution_from_band(80) == Critical", contribution_from_band(80) == ContributionLevel.CRITICAL)
_check("contribution_from_band(79.9) == High", contribution_from_band(79.9) == ContributionLevel.HIGH)
_check("contribution_from_band(19.9) == No Contribution", contribution_from_band(19.9) == ContributionLevel.NONE)

# 3. hard_stop must always imply part_score == 100 (the documented rule).
_check("Every hard_stop record has part_score == 100", bool((part_df.loc[part_df['hard_stop'], 'part_score'] == 100.0).all()))

# 4. Every hard_stop record must get Priority == Immediate.
_hs_record = part_df[part_df['hard_stop']].iloc[0]
_hs_report = generate_tprs_report(_hs_record)
_check("A hard_stop record is always assigned Priority = Immediate", _hs_report.priority == "Immediate")

assert _tests_passed, "One or more reliability checks failed -- investigate before trusting the output."

PASS  Axis-share decomposition reconstructs L and I exactly for every part
PASS  contribution_from_band(80) == Critical
PASS  contribution_from_band(79.9) == High
PASS  contribution_from_band(19.9) == No Contribution
PASS  Every hard_stop record has part_score == 100
PASS  A hard_stop record is always assigned Priority = Immediate


In [ ]:
# =============================================================================
# Batch-Level Completeness Audit & Dataset Composition Note
# =============================================================================

_total_high_risk = int(part_df["part_category"].isin(HIGH_RISK_CATEGORIES).sum())
_records_with_report = len(tprs_results)
_hard_stop_share = tprs_results["Hard Stop"].mean() if len(tprs_results) else 0

reliability_summary = pd.DataFrame([
    {"Check": "High/Critical parts in source data", "Result": _total_high_risk},
    {"Check": "Parts with a completed report", "Result": _records_with_report},
    {"Check": "Fully covered (no gaps)", "Result": _total_high_risk == _records_with_report},
    {"Check": "Share of processed parts that are hard-stop driven", "Result": f"{_hard_stop_share:.0%}"},
])
reliability_summary

,Check,Result
0,High/Critical parts in source data,31
1,Parts with a completed report,31
2,Fully covered (no gaps),True
3,Share of processed parts that are hard-stop driven,100%


In [ ]:
# Notable finding worth stating explicitly: in the current dataset, every
# Critical-category part is hard-stop driven -- the highest natural
# (non-hard-stop) score in the entire dataset tops out well below the
# Critical band. This is exactly why the hard-stop rule is treated as a
# first-class explanation in Section 11/13 rather than folded in as just
# another ranked factor.
_non_hard_stop_max = part_df.loc[~part_df["hard_stop"], "part_score"].max()
print(f"Highest part_score among non-hard-stop parts: {_non_hard_stop_max:.1f} "
      f"({classify(_non_hard_stop_max)})")
print(f"Hard-stop parts in this dataset: {int(part_df['hard_stop'].sum())} "
      f"(all classified Critical at exactly 100.0)")

Highest part_score among non-hard-stop parts: 55.3 (Medium)
Hard-stop parts in this dataset: 31 (all classified Critical at exactly 100.0)


## 19. LLM-Polished Narrative *(on by default)*

Everything above (Sections 1-18) is fully deterministic and remains the
system of record — this section never modifies `tprs_results` or the
file already exported in Section 17. It runs as an **additive** layer:
it takes the deterministic text and asks the model to rewrite it in
**plain, non-technical language** — short sentences, everyday words, no
jargon — so it's easier for a busy reader to understand at a glance.
The result is saved to its **own** file,
`total_part_risk_root_cause_analysis_llm_polished.csv`, so the
deterministic and LLM-polished outputs both exist side by side and
nothing is overwritten.

It runs on the **full batch** (every row in `tprs_results`) across the
three narrative fields — Human Explanation, Business Impact, and
Executive Summary.

Two safeguards, since simplifying language is exactly the kind of task
where an LLM might be tempted to also simplify away a number or
overstate a claim — the opposite of what "more reliable" should mean:

1. **Numeric integrity check** — every number in the original text must
   still be present in the polished text, or the polish is discarded and
   the deterministic original is kept for that field. Accuracy always
   wins over readability.
2. **Fallback on any failure** — a missing API key, no network access,
   or an API error falls back to the deterministic text for that field
   rather than crashing the notebook or leaving a blank cell.

Requires `pip install anthropic` and an `ANTHROPIC_API_KEY` environment
variable. Set `ENABLE_LLM_POLISH = False` below to skip this section
entirely and keep only the deterministic output from Section 17.

In [ ]:
# !pip install -q groq

In [ ]:
# IMPORTANT: LLM_MODEL must match the client instantiated in the cell
# below (currently anthropic.Anthropic()). If you swap in a different
# provider's model name (e.g. a Groq or OpenAI model like
# "llama-3.3-70b-versatile"), you must also swap the client instantiation
# a few cells down to that provider's SDK -- otherwise every call fails
# and safe_llm_polish() will silently fall back to the deterministic text
# for every record without any visible error beyond a logged warning.
POLISHED_EXPORT_FILE = OUTPUT_DIR / "total_part_risk_root_cause_analysis_polished.csv"

In [ ]:
# =============================================================================
# LLM Polishing Cell – using Groq (free, fast)
# =============================================================================

from groq import Groq
import re
import pandas as pd
from google.colab import userdata

# Retrieve Groq API key from secrets
API_KEY = userdata.get('GROQ_API_KEY')
if not API_KEY:
    raise ValueError("Groq API key not found. Add 'GROQ_API_KEY' to Colab secrets.")

client = Groq(api_key=API_KEY)
MODEL = "llama-3.3-70b-versatile"   # or "mixtral-8x7b-32768"

def polish_tprs_text(row: pd.Series) -> dict:
    """
    Rephrase the four text columns according to user rules.
    """
    prompt = f"""
    You are a senior supply chain communication specialist.

    Rewrite the following four sections of a risk report in a professional, business‑friendly tone suitable for procurement and supply chain managers.

    Follow these rules strictly:
    - Start with the overall conclusion in one sentence.
    - Explain why the part reached this risk level (use the root causes and scores naturally).
    - Explain the business impact rather than only the technical reason.
    - Mention important numerical scores naturally (e.g., 'score of 80 out of 100').
    - Do NOT repeat the same information.
    - Do NOT mention internal implementation details like 'hard‑stop rule' – if it applies, describe it as 'a project risk policy that overrides the score when specific conditions are met'.
    - End with a clear, actionable recommendation.
    - Keep the entire rewritten version between 120 and 170 words.
    - Use a professional, concise tone.

    Output each section clearly labelled with the headers:

    1. Human Explanation
    2. Business Impact
    3. Executive Summary
    4. Recommendations

    **Original text:**

    Human Explanation: {row['Human Explanation']}

    Business Impact: {row['Business Impact']}

    Executive Summary: {row['Executive Summary']}

    Recommendations: {row['Recommendations']}

    ---
    Rewritten version (use the same headers):
    """
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            max_tokens=700
        )
        output = response.choices[0].message.content.strip()

        # Parse the output into sections
        parsed = {}
        for section in ["Human Explanation", "Business Impact", "Executive Summary", "Recommendations"]:
            pattern = rf"{section}:\s*(.*?)(?=\n(?:Human Explanation|Business Impact|Executive Summary|Recommendations|$))"
            match = re.search(pattern, output, re.DOTALL | re.IGNORECASE)
            if match:
                parsed[section] = match.group(1).strip()
            else:
                parsed[section] = row[section]
        return parsed
    except Exception as e:
        print(f"⚠️ LLM error for row {row.name}: {e}. Keeping original.")
        return {
            "Human Explanation": row["Human Explanation"],
            "Business Impact": row["Business Impact"],
            "Executive Summary": row["Executive Summary"],
            "Recommendations": row["Recommendations"]
        }

# ---- Load the exported CSV ----
df = pd.read_csv(EXPORT_FILE, encoding="utf-8")

# ---- Process sample or all rows ----
SAMPLE_SIZE = None    # change to None to process all
rows_to_process = df.head(SAMPLE_SIZE) if SAMPLE_SIZE else df

for idx, row in rows_to_process.iterrows():
    if row["Part Category"] in ["High", "Critical"]:
        if SAMPLE_SIZE and idx >= SAMPLE_SIZE:
            break
        polished = polish_tprs_text(row)
        df.at[idx, "Human Explanation"] = polished["Human Explanation"]
        df.at[idx, "Business Impact"] = polished["Business Impact"]
        df.at[idx, "Executive Summary"] = polished["Executive Summary"]
        df.at[idx, "Recommendations"] = polished["Recommendations"]

# ---- Save the polished CSV (overwrite) ----
df.to_csv(POLISHED_EXPORT_FILE, index=False, encoding="utf-8-sig")
print(f"✅ Polished CSV saved to: {POLISHED_EXPORT_FILE}")

✅ Polished CSV saved to: outputs/total_part_risk_root_cause_analysis_polished.csv


In [ ]:
# =============================================================================
# Create a simplified user‑friendly CSV: Merge Human Explanation, Business Impact, Executive Summary
# =============================================================================

# Choose your source file – change to POLISHED_EXPORT_FILE if you have an LLM‑polished version
SOURCE_FILE = POLISHED_EXPORT_FILE   # or POLISHED_EXPORT_FILE

# Load the CSV
df = pd.read_csv(SOURCE_FILE, encoding="utf-8")

# Merge the three columns into a single "Root Cause Analysis" column
df["Root Cause Analysis"] = (
    "📌 **Explanation**:\n" + df["Human Explanation"] + "\n\n" +
    "💼 **Business Impact**:\n" + df["Business Impact"] + "\n\n" +
    "📊 **Executive Summary**:\n" + df["Executive Summary"]
)

# Keep only the columns you want to show
columns_to_keep = [
    "MPN",
    "Part Score",
    "Priority",
    "Part Category",
    "Hard Stop",
    "Score Detected Date",
    "Human Explanation",
    "Business Impact",
    "Executive Summary",
    "Root Cause Analysis",
    "Recommendations",
    "Score Detected Date"   # if you want to keep the separate Business Impact column
]
# If you prefer to have only the merged column and drop the separate Business Impact, remove it from the list.

df_simple = df[columns_to_keep].copy()

# Save with UTF‑8‑SIG for Excel compatibility
SIMPLE_OUTPUT = OUTPUT_DIR / "total_part_risk_root_cause_analysis_simple.csv"
df_simple.to_csv(SIMPLE_OUTPUT, index=False, encoding="utf-8-sig")

print(f"✅ Simplified CSV saved to: {SIMPLE_OUTPUT}")

✅ Simplified CSV saved to: outputs/total_part_risk_root_cause_analysis_simple.csv


## Summary

- **Input**: the TPRS v4 output (`N_STOCK`, `N_MFR`, `N_ALT`, `N_COMP`,
  `L`, `I`, `part_score`, `part_category`, `hard_stop`), reused exactly
  as produced upstream, enriched with real Stock and Alternative detail.
  Scoped strictly to the four factors TPRS is built from — no other
  dataset is referenced.
- **Formula fidelity**: AHP weights are re-derived from the source
  notebook's exact pairwise matrices (Section 7) and verified to
  reconstruct `L` and `I` exactly for every record (Section 18) — not
  approximated or guessed.
- **Hard-stop rule**: treated as a first-class explanation, not a ranked
  contributor, because every Critical part in this dataset is hard-stop
  driven — confirmed, not assumed (Section 18).
- **Reliability**: a fully deterministic core (Sections 1-18, always the system of record) plus an LLM polish layer (Section 19, on by default) that runs on the full batch, guards every field with a numeric-integrity check, and saves to its own file rather than overwriting the deterministic export.
- **Output**: `outputs/total_part_risk_root_cause_analysis.csv` (deterministic, Section 17), one row
  per High/Critical part, with MPN, Part Score, Part Category, Hard
  Stop, Priority, Score Detected Date, Detected Root Causes, Human
  Explanation, Business Impact, Executive Summary, and Recommendations
  — in business language, grounded in the actual extracted formula. A second file, `outputs/total_part_risk_root_cause_analysis_llm_polished.csv` (Section 19), adds an LLM-polished version of each narrative field alongside the original.